# Synthetic experiment validation and failure-mode checks

**Research question:** Can the analysis pipeline detect healthy randomization and reject an SRM-contaminated run?

This notebook is the review interface. Reusable transformations remain in `src/`, `scripts/`, and `sql/`; the notebook reads their committed evidence rather than creating a second implementation. Raw source data is intentionally not embedded.

## Hypotheses declared before interpretation

- Healthy assignment should pass SRM and covariate balance checks.
- The deliberately broken run should be rejected before effect interpretation.
- Injected treatment effects validate code, not product impact.

In [1]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
print(f'Repository: {ROOT.name}')


Repository: shopflow-product-analytics


## Feature provenance

Every row states its raw source, transformation, decision-time cutoff, fit scope, missing-value treatment, and known leakage or proxy risk.

In [2]:
features = pd.read_csv(ROOT / 'reports' / 'feature_provenance.csv')
display(features)
display(features.groupby(['feature_family', 'model_role']).size().rename('features').reset_index())


,feature_name,feature_family,raw_source,grain,transformation,observation_cutoff,available_at_decision,fit_partition,missing_treatment,model_role,exclusion_reason,leakage_risk,proxy_or_fairness_risk,validation
0,prior_sessions,history,mart_session_funnel,user,Count sessions before first_eligible_at,strictly before eligible session,yes,none,zero,experiment covariate,NaN,low,none identified,point-in-time SQL predicate and unit tests
1,prior_cart_sessions,history,mart_session_funnel.reached_cart,user,Count prior sessions reaching cart,strictly before eligible session,yes,none,zero,audit,NaN,low,none identified,point-in-time SQL predicate
2,prior_purchase_sessions,history,mart_session_funnel.reached_purchase,user,Count prior sessions reaching purchase,strictly before eligible session,yes,none,zero,experiment covariate,NaN,low,may proxy relationship tenure,covariate balance SMD
3,prior_purchase_item_value,history,mart_session_funnel.purchase_item_value,user,Sum prior purchase item value,strictly before eligible session,yes,none,zero,CUPED covariate,NaN,low,spend may proxy customer value,CUPED precision comparison
4,prior_purchase_rate,history,prior purchase and session counts,user,prior_purchase_sessions / prior_sessions,strictly before eligible session,yes,none,zero when no history,experiment covariate,NaN,low,may proxy tenure,covariate balance SMD
5,returning_user,history,prior_sessions,user,1 when prior_sessions > 0,strictly before eligible session,yes,none,zero,heterogeneity slice,NaN,low,new-user status may proxy acquisition channel,pre-declared subgroup readout
6,price_quartile,product context,representative_price,session-product,Portfolio quantile bucket Q1-Q4,current viewed/carted product,yes,analysis population,unknown,association model,NaN,medium: conditions on carted products,price can proxy assortment/customer mix,adjusted GLM; non-causal interpretation
7,category_level_1,product context,category_code,session-product,Top-level category parsed from taxonomy,current product,yes,none,unknown,association model,NaN,low,category mix,cluster-robust GLM
8,activity_month,time control,session_started_at,session-product,YYYY-MM calendar month,current session,yes,none,unknown,association control,NaN,medium if used across future periods,seasonality,included as control only
9,converted_same_product,outcome,ordered full-funnel events,user/eligible session,Any ordered full-funnel product in eligible se...,after assignment,yes,none,false,target only,NaN,high if used as predictor,none,kept out of feature matrix


,feature_family,model_role,features
0,history,CUPED covariate,1
1,history,audit,1
2,history,experiment covariate,3
3,history,heterogeneity slice,1
4,outcome,target only,1
5,product context,association model,2
6,time control,association control,1


## Decision-time audit

A feature is eligible only when its information exists at the stated observation cutoff. Model-fitted bins, encoders, imputers, and thresholds use development data only. Outcomes and evaluation groups never enter the predictor matrix.

In [3]:
audit = features[['feature_name', 'observation_cutoff', 'available_at_decision', 'fit_partition', 'leakage_risk', 'validation']]
display(audit)


,feature_name,observation_cutoff,available_at_decision,fit_partition,leakage_risk,validation
0,prior_sessions,strictly before eligible session,yes,none,low,point-in-time SQL predicate and unit tests
1,prior_cart_sessions,strictly before eligible session,yes,none,low,point-in-time SQL predicate
2,prior_purchase_sessions,strictly before eligible session,yes,none,low,covariate balance SMD
3,prior_purchase_item_value,strictly before eligible session,yes,none,low,CUPED precision comparison
4,prior_purchase_rate,strictly before eligible session,yes,none,low,covariate balance SMD
5,returning_user,strictly before eligible session,yes,none,low,pre-declared subgroup readout
6,price_quartile,current viewed/carted product,yes,analysis population,medium: conditions on carted products,adjusted GLM; non-causal interpretation
7,category_level_1,current product,yes,none,low,cluster-robust GLM
8,activity_month,current session,yes,none,medium if used across future periods,included as control only
9,converted_same_product,after assignment,yes,none,high if used as predictor,kept out of feature matrix


## Committed analytical evidence

In [4]:
evidence_files = ["reports/generated/experiment_healthy_results.json", "reports/generated/experiment_srm_results.json"]
for relative in evidence_files:
    path = ROOT / relative
    display(Markdown(f'### `{relative}`'))
    if not path.exists():
        display(Markdown('*Not committed; reproduce locally using the README pipeline.*'))
    elif path.suffix == '.csv':
        display(pd.read_csv(path).head(20))
    elif path.suffix == '.json':
        payload = json.loads(path.read_text())
        display(payload)


### `reports/generated/experiment_healthy_results.json`

{'scenario': 'healthy',
 'evidence_class': 'simulation_only',
 'outcome_provenance': 'synthetic_injected_effect',
 'unique_source_users': 9742,
 'duplicate_source_users': 0,
 'injected_treatment_log_odds': {'base': 0.12,
  'minimum': 0.12,
  'maximum': 0.14159999999999998},
 'sample_size': 9742,
 'assignment': {'control': 4862, 'treatment': 4880},
 'srm_p_value': 0.855294000505257,
 'max_absolute_covariate_smd': 0.015536611545388582,
 'covariate_smd': {'prior_sessions': 0.015536611545388582,
  'prior_purchase_rate': 0.014880818922852562,
  'prior_purchase_item_value': -0.006622856639334979},
 'primary_conversion': {'control_rate': 0.4370629370629371,
  'treatment_rate': 0.46577868852459015,
  'absolute_effect': 0.02871575146165306,
  'relative_lift': 0.06570163934426221,
  'ci_low': 0.008952152191702725,
  'ci_high': 0.04844735450125667,
  'p_value': 0.004403045962033357},
 'purchase_value_per_eligible_user': {'difference': 4.983043705047521,
  'ci_low': -1.6040523118782142,
  'ci_high

### `reports/generated/experiment_srm_results.json`

{'scenario': 'srm',
 'evidence_class': 'simulation_only',
 'outcome_provenance': 'synthetic_injected_effect',
 'unique_source_users': 9162,
 'duplicate_source_users': 0,
 'injected_treatment_log_odds': {'base': 0.12,
  'minimum': 0.12,
  'maximum': 0.14159999999999998},
 'sample_size': 9162,
 'assignment': {'control': 4843, 'treatment': 4319},
 'srm_p_value': 4.3900844411794736e-08,
 'max_absolute_covariate_smd': 0.01743685519364015,
 'covariate_smd': {'prior_sessions': 0.01743685519364015,
  'prior_purchase_rate': -0.0011432103986936358,
  'prior_purchase_item_value': -0.009656172970990473},
 'primary_conversion': {'control_rate': 0.43836465001032415,
  'treatment_rate': 0.46885853206760825,
  'absolute_effect': 0.030493882057284094,
  'relative_lift': 0.06956282185747842,
  'ci_low': 0.010073844546555372,
  'ci_high': 0.05088738770482679,
  'p_value': 0.0034208251829590305},
 'purchase_value_per_eligible_user': {'difference': 5.6508075659026105,
  'ci_low': -1.1535569395601826,
  'ci

## Decision

**The synthetic run validates the measurement pipeline only; a real powered randomized test is still required.**

### Claim boundary

The notebook distinguishes observed evidence from model output, simulated policy, and causal impact. A strong offline metric is not production exposure.